In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from tqdm import tqdm
import torch
import torch.nn.functional as F
import numpy as np

from encoder import ConvEncoder
from contextualizer import Contextualizer
from foundationModelTrainer import BENDRTrainer

SyntaxError: unexpected EOF while parsing (contextualizer.py, line 146)

In [5]:
pretrain_args = {
  "mask_rate": 0.065,
  "mask_span": 10,
  "layer_drop": 0.01,
  "multi_gpu": True,
  "temp": 0.1,
  "encoder_grad_frac": 0.1,
  "num_negatives": 20,
  "enc_feat_l2": 1.0
}

In [21]:
encoder = ConvEncoder(19, encoder_h=512)
contextualizer = Contextualizer(encoder.encoder_h, layer_drop=0.01)
trainer = BENDRTrainer(encoder, contextualizer, **pretrain_args)

In [4]:
# Slower learning rate for the encoder
process.set_optimizer(torch.optim.Adam(process.parameters()))
process.add_batch_transform(RandomTemporalCrop(max_crop_frac=0.05))

In [9]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC.pt'), weights_only=True)
        eo = torch.load(os.path.join(path, subject, 'EO.pt'), weights_only=True)

        ecs.append(ec)
        eos.append(eo)

    ecs = torch.cat(ecs, axis=0).float()
    eos = torch.cat(eos, axis=0).float()

    print(ecs.shape, eos.shape)

    ecs_dataset = torch.utils.data.TensorDataset(ecs)
    eos_dataset = torch.utils.data.TensorDataset(eos)

    return ecs_dataset, eos_dataset

In [10]:
ecs_dataset, eos_dataset = load_dataset('/home/azureuser/mycontainer/SmallSubset')

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/20 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 70.28it/s]


torch.Size([60, 19, 15360]) torch.Size([20, 19, 15360])


In [11]:
print(ecs_dataset[0][0].shape, ecs_dataset[0][0].dtype)

torch.Size([19, 15360]) torch.float32


In [12]:
process.fit(ecs_dataset, epochs=5, batch_size=8, validation_dataset=eos_dataset)

Loading data with 4 additional workers


Epoch:  20%|##        | 1/5 [00:00<?, ?epoch/s]

Iteration:  14%|#4        | 1/7 [00:00<?, ?batches/s]

/anaconda/envs/EEG-ML/lib/python3.8/site-packages/torch/nn/functional.py:1381: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


Training: End of Epoch 1 | Accuracy: 1.16% | Mask_pct: 44.81% | loss: 4.354 | lr: 6.567e-04 | epoch: 1.000 |


...

Training: End of Epoch 5 | Accuracy: 3.29% | Mask_pct: 44.75% | loss: 3.867 | lr: 2.472e-05 | epoch: 5.000 |


Predicting:   0%|          | 0/3 [00:00<?, ?it/s]

/anaconda/envs/EEG-ML/lib/python3.8/site-packages/torch/nn/functional.py:1381: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


Validation: End of Epoch 5 | Accuracy: 0.47% | Mask_pct: 30.00% | loss: 4.086 |
Best loss. Retaining checkpoint...
Loading best model...


(    Accuracy  Mask_pct      loss            lr  epoch  iteration
 0   0.000000  0.433594  5.157291  1.043078e-04      1          1
 1   0.000781  0.410156  4.687973  2.800000e-04      1          2
 2   0.007031  0.444531  4.377607  5.200000e-04      1          3
 3   0.017188  0.447656  4.207792  7.600000e-04      1          4
 4   0.016406  0.471094  4.000239  9.356922e-04      1          5
 5   0.021094  0.463281  4.071789  1.000000e-03      1          6
 6   0.018750  0.466406  3.978347  9.968561e-04      1          7
 7   0.009375  0.463281  3.945912  9.874640e-04      2          1
 8   0.013281  0.514063  3.885136  9.719418e-04      2          2
 9   0.017969  0.471094  3.946582  9.504846e-04      2          3
 10  0.032812  0.474219  3.892948  9.233624e-04      2          4
 11  0.025781  0.456250  3.910646  8.909162e-04      2          5
 12  0.013281  0.422656  3.966225  8.535540e-04      2          6
 13  0.014844  0.485156  3.872805  8.117457e-04      2          7
 14  0.004